# Notebook de Avaliação do Modelo de Crédito

Este notebook consome a API de monitoramento em `http://localhost:8001` para avaliar o modelo em dois aspectos:

- **Performance**: AUC-ROC e volumetria mensal de registros do arquivo `batch_records.json`
- **Aderência**: teste de Kolmogorov-Smirnov comparando a distribuição de scores de novos datasets com a base de referência (`test.gz`)

> Antes de executar, certifique-se de que a API está rodando: `uvicorn app.main:app --port 8001`

In [12]:
# Imports e configuração de caminhos
import json
import os
from pathlib import Path

import pandas as pd
import requests

# Detecta o diretório do notebook independentemente de onde o Jupyter foi iniciado
_cwd = Path(os.getcwd())
MONITORING_DIR = _cwd if _cwd.name == "monitoring" else _cwd / "monitoring"
PROJECT_DIR = MONITORING_DIR.parent

BASE_URL = "http://localhost:8001/v1"

print(f"Diretório monitoring : {MONITORING_DIR}")
print(f"Raiz do projeto      : {PROJECT_DIR}")
print(f"API base URL         : {BASE_URL}")

Diretório monitoring : /Users/ladainha/Documents/B3/monitoramento-modelos-ml/monitoring
Raiz do projeto      : /Users/ladainha/Documents/B3/monitoramento-modelos-ml
API base URL         : http://localhost:8001/v1


---
## 1. Avaliação de Performance

Carrega os registros de `batch_records.json` e envia para `POST /v1/performance`.
A resposta retorna:
- **Volumetria**: quantidade de registros agrupada por mês (`REF_DATE`)
- **AUC-ROC**: métrica de discriminação do modelo aplicada a esses registros, usando `TARGET` como gabarito

In [13]:
# Carrega o arquivo de registros em lote
batch_path = MONITORING_DIR / "batch_records.json"
with open(batch_path) as f:
    records = json.load(f)

print(f"Total de registros carregados: {len(records)}")

# Envia os registros para o endpoint de performance e valida a resposta
response_perf = requests.post(
    f"{BASE_URL}/performance",
    json={"records": records},
    timeout=60,
)
response_perf.raise_for_status()
perf = response_perf.json()

print("Resposta recebida com sucesso.")

Total de registros carregados: 500
Resposta recebida com sucesso.


In [9]:
# Exibe a volumetria de registros por mês como tabela ordenada
print("=== Volumetria por mês ===")
vol_df = (
    pd.DataFrame(perf["volumetria"].items(), columns=["Mês", "Quantidade"])
    .sort_values("Mês")
    .reset_index(drop=True)
)
display(vol_df)

# Exibe o AUC-ROC calculado pelo modelo sobre os registros enviados
print(f"\n=== AUC-ROC ===")
print(f"AUC-ROC: {perf['auc_roc']:.4f}")

=== Volumetria por mês ===


,Mês,Quantidade
0,2017-01,58
1,2017-02,55
2,2017-03,62
3,2017-04,49
4,2017-05,67
5,2017-06,63
6,2017-07,74
7,2017-08,72



=== AUC-ROC ===
AUC-ROC: 0.5752


---
## 2. Aderência — base de treino (`train.gz`)

Verifica se a distribuição de scores do dataset de **treino** é estatisticamente semelhante
à distribuição de scores da base de referência (`test.gz`) usando o teste de Kolmogorov-Smirnov.

- **KS Statistic** próximo de 0 → distribuições similares (boa aderência)
- **P-valor** alto (> 0,05) → não há evidência estatística de diferença entre as distribuições

In [14]:
# Constrói o caminho absoluto para o dataset de treino
train_path = str((PROJECT_DIR / "datasets/credit_01/train.gz").resolve())
print(f"Dataset enviado: {train_path}")

# Envia o caminho para o endpoint de aderência
# O servidor lê o arquivo, escora com o modelo e calcula o KS vs test.gz
response_train = requests.post(
    f"{BASE_URL}/aderencia",
    json={"dataset_path": train_path},
    timeout=120,
)
response_train.raise_for_status()
ader_train = response_train.json()

# Exibe os resultados do teste KS
print("\n=== Aderência: train.gz vs test.gz ===")
print(f"KS Statistic : {ader_train['ks_statistic']:.4f}")
print(f"P-valor      : {ader_train['p_value']:.4f}")

Dataset enviado: /Users/ladainha/Documents/B3/monitoramento-modelos-ml/datasets/credit_01/train.gz

=== Aderência: train.gz vs test.gz ===
KS Statistic : 0.0019
P-valor      : 0.9993


---
## 3. Aderência — base out-of-time (`oot.gz`)

Repete a análise para o dataset **out-of-time** (`oot.gz`), que representa dados de um período
posterior ao treinamento do modelo.

Um KS maior do que o observado na base de treino pode indicar **data drift** ao longo do tempo,
sinalizando degradação potencial na performance do modelo em produção.

In [15]:
# Constrói o caminho absoluto para o dataset out-of-time
oot_path = str((PROJECT_DIR / "datasets/credit_01/oot.gz").resolve())
print(f"Dataset enviado: {oot_path}")

# Envia o caminho para o endpoint de aderência
response_oot = requests.post(
    f"{BASE_URL}/aderencia",
    json={"dataset_path": oot_path},
    timeout=120,
)
response_oot.raise_for_status()
ader_oot = response_oot.json()

# Exibe os resultados do teste KS
print("\n=== Aderência: oot.gz vs test.gz ===")
print(f"KS Statistic : {ader_oot['ks_statistic']:.4f}")
print(f"P-valor      : {ader_oot['p_value']:.4f}")

Dataset enviado: /Users/ladainha/Documents/B3/monitoramento-modelos-ml/datasets/credit_01/oot.gz

=== Aderência: oot.gz vs test.gz ===
KS Statistic : 0.0217
P-valor      : 0.0000
